# 第24章　説明可能性（XAI）の深掘り ― 手法と、その落とし穴**『医療診断支援AIの社会実装（社会実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-social

## Grad-CAMを、実装の骨格で

In [ ]:
# Grad-CAMの骨格acts = {}     # 最終conv層の出力を保存grads = {}    # そこへの勾配を保存target_layer.register_forward_hook(lambda m,i,o: acts.update(v=o))target_layer.register_full_backward_hook(lambda m,gi,go: grads.update(v=go[0]))score = model(x)[0, target_class]     # 対象クラスのスコアscore.backward()w = grads["v"].mean(dim=(2, 3))       # 各特徴マップの重み=勾配の空間平均cam = (w[..., None, None] * acts["v"]).sum(1)cam = cam.clamp(min=0)                # 正の寄与だけ（ReLU）# 正の寄与が一つも無い入力では cam.max()==0 になり、割ると NaN が出る。# また、バッチ全体の最大値で割ると症例間で基準が動くので、症例ごとに正規化する。peak = cam.amax(dim=(1, 2), keepdim=True)                  # (B,1,1)has_positive = peak > 0                                     # 全ゼロのマップを区別するcam = torch.where(has_positive, cam / peak.clamp_min(1e-12), torch.zeros_like(cam))# 全ゼロのマップを「病変なし」「安全」と読まないこと。別の状態として表示する。

## 「似た症例」を根拠に添える ― 検索に基づく説明

In [ ]:
# 過去の確定症例embeddingの索引から、似たk件を引いて根拠にするq = encoder(current).embeddingidx = ann_index.search(q, k=5)                 # 近傍探索（同一患者は除外＝リーク防止）neighbors = [db[i] for i in idx]evidence = [(n.diagnosis, n.similarity, n.image_ref) for n in neighbors]vote = majority([n.diagnosis for n in neighbors])   # 検索に基づく第二の意見